Import Libraries

In [ ]:
# https://github-com.translate.goog/opendatalab/MinerU/blob/master/README_zh-CN.md?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=en&_x_tr_pto=wapp
# https://github.com/InternLM/InternLM

In [ ]:
# Delete headers, footers, footnotes, page numbers and other elements to ensure semantic coherence
# Output text in human reading order, suitable for single column, multi-column and complex layout
# Preserve the structure of the original document, including titles, paragraphs, lists, etc.
# Extract images, image descriptions, tables, table titles and footnotes
# Automatically recognize and convert formulas in documents to LaTeX format
# Automatically recognize and convert tables in documents to HTML format
# Automatically detect scanned PDFs and garbled PDFs, and enable OCR function
# OCR supports detection and recognition of 84 languages
# Supports multiple output formats, such as multimodal and NLP Markdown, JSON sorted by reading order, intermediate formats containing rich information, etc.
# Supports a variety of visualization results, including layout visualization, span visualization, etc., to facilitate efficient confirmation of output effects and quality inspection
# Supports pure CPU environment operation and GPU (CUDA) / NPU (CANN) / MPS acceleration

In [1]:
!python3 --version

Python 3.10.12


In [ ]:
# PLEASE use env

In [6]:
# !conda create -n MinerU python=3.10
# !conda activate MinerU
# !pip install -U "magic-pdf[full]" --extra-index-url https://wheels.myhloli.com -i https://mirrors.aliyun.com/pypi/simple

In [ ]:
# https://github-com.translate.goog/opendatalab/MinerU/blob/master/docs/how_to_download_models_zh_cn.md?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=en&_x_tr_pto=wapp

In [ ]:
# wget https://gcore.jsdelivr.net/gh/opendatalab/MinerU@master/scripts/download_models.py -O download_models.py


In [ ]:
# python download_models.py

In [9]:
# !pip install modelscope

In [3]:
# pip install -U magic-pdf[full] --extra-index-url https://wheels.myhloli.com

In [2]:
!magic-pdf --version

/bin/bash: line 1: magic-pdf: command not found


In [25]:

# Path to the PDF
pdf_path = "/home/recoil/shared_space/parserBibliographyJT/sample_literature_Freight/2023_Nassar_SystemDynamicsModel.pdf"

# Extract the base name of the PDF file (without extension) for folder and file names
pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]

# Directory for storing the converted page images
output_pages_dir = f"{pdf_name}_pages"

# Create the directory for page images if it doesn't exist
os.makedirs(output_pages_dir, exist_ok=True)

# Open the PDF document
doc = fitz.open(pdf_path)

# Define a high resolution factor (higher values = better quality)
zoom_factor = 3  # You can increase this value for even higher quality (e.g., 5, 6, etc.)

# Convert each page to an image
for page_number in range(len(doc)):
    page = doc[page_number]

    # Set the zoom factor for the highest image quality
    matrix = fitz.Matrix(zoom_factor, zoom_factor)  # Apply zoom factor to render high quality

    # Render the page as a pixmap (image) with the high resolution
    pix = page.get_pixmap(matrix=matrix)

    # Define the output image filename
    image_filename = os.path.join(output_pages_dir, f"page_{page_number+1}.png")

    # Save the image
    pix.save(image_filename)
    print(f"Saved page as image: {image_filename}")

print(f"All pages have been converted to images and saved in: {output_pages_dir}")


Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_1.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_2.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_3.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_4.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_5.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_6.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_7.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_8.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_9.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_10.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_11.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_12.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_13.png
Saved page as image: 2023_Nassar_SystemDynamicsModel_pages/page_14.png
Saved page as i

In [19]:
import cv2
from doclayout_yolo import YOLOv10
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download


# Load the pre-trained model
filepath = hf_hub_download(repo_id="juliozhao/DocLayout-YOLO-DocStructBench", filename="doclayout_yolo_docstructbench_imgsz1024.pt")
model = YOLOv10(filepath)
# model = YOLOv10.from_pretrained("juliozhao/DocLayout-YOLO-DocStructBench")#YOLOv10("path/to/provided/model")



In [ ]:
#https://github.com/opendatalab/DocLayout-YOLO

In [21]:
import os
import cv2

# Define the folder containing images and the output folder for annotated images
input_folder = "/home/recoil/shared_space/parserBibliographyJT/2023_Nassar_SystemDynamicsModel_pages"
output_folder = "/home/recoil/shared_space/parserBibliographyJT/Annotated_Results"

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Iterate through all files in the input folder
for filename in os.listdir(input_folder):
    # Construct the full path of the image
    file_path = os.path.join(input_folder, filename)

    # Check if it's an image file (you can refine this condition if needed)
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):
        print(f"Processing: {filename}")

        # Perform prediction on the image
        det_res = model.predict(
            file_path,
            imgsz=1024,        # Prediction image size
            conf=0.2,          # Confidence threshold
            device="cuda:0"    # Device to use (e.g., 'cuda:0' or 'cpu')
        )

        # Annotate the result
        annotated_frame = det_res[0].plot(line_width=5, font_size=20)  # Returns a NumPy array

        # Save the annotated image to the output folder
        output_path = os.path.join(output_folder, f"annotated_{filename}")
        cv2.imwrite(output_path, annotated_frame)  # Use OpenCV to save the NumPy array as an image

        print(f"Saved: {output_path}")


Processing: page_7.png

image 1/1 /home/recoil/shared_space/parserBibliographyJT/2023_Nassar_SystemDynamicsModel_pages/page_7.png: 1024x768 4 plain texts, 4 abandons, 2 figures, 2 figure_captions, 18.2ms
Speed: 3.3ms preprocess, 18.2ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 768)
Saved: /home/recoil/shared_space/parserBibliographyJT/Annotated_Results/annotated_page_7.png
Processing: page_17.png

image 1/1 /home/recoil/shared_space/parserBibliographyJT/2023_Nassar_SystemDynamicsModel_pages/page_17.png: 1024x768 1 title, 10 plain texts, 4 abandons, 3 figures, 2 figure_captions, 2 tables, 2 table_captions, 18.2ms
Speed: 3.3ms preprocess, 18.2ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 768)
Saved: /home/recoil/shared_space/parserBibliographyJT/Annotated_Results/annotated_page_17.png
Processing: page_3.png

image 1/1 /home/recoil/shared_space/parserBibliographyJT/2023_Nassar_SystemDynamicsModel_pages/page_3.png: 1024x768 2 titles, 13 plain texts, 3 a